# Scholl et al. (2015) on CRCNS pvc-10

Paper: Scholl, Pattadkal, Dilly, Priebe and Zemelman (2015), "Local Integration Accounts for Weak
Selectivity of Mouse Neocortical Parvalbumin Interneurons", Neuron 87:424-436,
doi:10.1016/j.neuron.2015.06.030. Claims file: `claims/scholl-2015.yaml`, written from the version
of record (main text, Experimental Procedures and Supplemental Information legends).

Data set: CRCNS pvc-10, "Disparity selectivity in mouse V1 measured using two-photon calcium
imaging", doi:10.6080/K0GT5K3V, read anonymously from the public S3 bucket `crcnsarchive`.

Version. CRCNS data sets are not versioned, so the version is recorded as `unversioned`. The files
read are those of the data set's published folder `pvc-10/live/`, the only folder its file list
(`live/filelist.txt`) and its `live/checksums.md5` describe; the MD5 sums of every file read are
printed below, so that a later change is detectable. The bucket also holds a folder
`pvc-10/original/to_remove/` with older processed files that are in neither list and are marked for
removal. They are not a published version and cannot be pinned, so no result here uses them; the
notebook only lists what they hold, as a finding about the data set.

Outcome in brief. The published folder holds the raw two-photon movies (green and red channels),
the stimulus tables and per-mouse plane shifts of 6 PV-Cre;Ai14 mice, and 50 example cell masks for
one plane. It holds no cell masks for the paper's 4,320 cells, no PV+/PV- labels, no responses or
measures per cell, no neuropil patches, no histology and no CaMKIIa animals. Every claim rests on
the paper's hand-identified cells or on data not deposited, so every claim is blocked.

| Claim | Outcome | Reason |
|---|---|---|
| fig1e-pv-tdtomato-colocalization | blocked | data_missing: no histology in the data set |
| cell-counts-responsive | blocked | data_missing: the paper's cell masks and PV+/PV- labels are not deposited |
| fig3abc-pv-more-binocular-fraction | blocked | data_missing, as above |
| fig3c-contralateral-bias | blocked | data_missing, as above |
| fig3-pv-lower-monocularity | blocked | data_missing, as above |
| mi-per-animal | blocked | data_missing, as above |
| fig3def-pv-weaker-disparity-selectivity | blocked | data_missing, as above |
| mi-dsi-relation | blocked | data_missing, as above |
| pv-larger-dff-low-dsi | blocked | data_missing, as above |
| figs3-pv-distinct-from-neuropil | blocked | data_missing: cells and neuropil patches not deposited |
| fig4ag-salt-and-pepper | blocked | data_missing, as above |
| fig4d-pv-od-matches-local-50um | blocked | data_missing, as above |
| fig4f-pv-od-spatial-decay | blocked | data_missing, as above |
| fig4f-od-length-constant | blocked | data_missing, as above |
| fig4ef-pvminus-od-unrelated | blocked | data_missing, as above |
| figs6ab-pvminus-pv-like-od | blocked | data_missing, as above |
| figs4-same-plane-control | blocked | data_missing, as above |
| figs5-neuropil-no-relation | blocked | data_missing: neuropil patches not deposited |
| fig5-camkii-responsive | blocked | data_missing: the CaMKIIa animals are not in the data set |
| fig5e-camkii-od-unrelated | blocked | data_missing, as above |
| fig4i-pv-disparity-matches-local | blocked | data_missing: cells and labels not deposited |
| fig4k-pv-disparity-length-constant | blocked | data_missing, as above |
| fig4jk-pvminus-disparity-weak | blocked | data_missing, as above |
| figs6cd-pvminus-pv-like-disparity | blocked | data_missing, as above |
| fig5f-camkii-disparity-unrelated | blocked | data_missing: the CaMKIIa animals are not in the data set |


In [ ]:
import collections
import hashlib
import resource
import time

import matplotlib.pyplot as plt
import numpy as np

import pvc10

T0 = time.time()
print("dataset", pvc10.DATASET, pvc10.VERSION, "doi", pvc10.DOI, "folder", pvc10.LIVE)

## Data check

### Files in the bucket

The data descriptor (live/docs, Version 0.8, April 8, 2016) says the data are from 6 animals in
folders mouse5 to mouse10, with green and red channel movies and a stimulus file per plane and a
stats file per mouse. It also says "The dataset provided here contains information about the
fluorescence traces, peak responses per condition, the measured binocular characteristics for each
cell like its disparity selectivity index, preferred relative disparity, the position of the cell in
the imaging volume". The listing below checks what is there.

In [ ]:
listing = pvc10.list_bucket()
by_folder = collections.defaultdict(lambda: [0, 0])
for k, s in listing:
    if k.endswith("/"):
        continue
    parts = k.split("/")
    folder = "/".join(parts[1:3]) if parts[1] == "live" else "/".join(parts[1:3])
    if parts[1] == "live" and parts[2] == "data":
        folder = "live/data/" + parts[3]
    by_folder[folder][0] += 1
    by_folder[folder][1] += s
print(f"{'folder':28s} {'files':>6s} {'GB':>8s}")
for f, (n, s) in sorted(by_folder.items()):
    print(f"{f:28s} {n:6d} {s / 1e9:8.2f}")
live_data = [(k, s) for k, s in listing if k.startswith(pvc10.LIVE + "data/") and not k.endswith("/")]
print("live/data total", round(sum(s for _, s in live_data) / 1e9, 1), "GB in", len(live_data), "files")

kinds = collections.Counter()
for k, _ in live_data:
    name = k.rsplit("/", 1)[1]
    kind = ("green" if name.endswith("green.mat") else "red" if name.endswith("red.mat")
            else "stim" if name.endswith("stim.mat") else "stats" if name.endswith("stats.mat") else "other")
    kinds[kind] += 1
print("live/data files by kind", dict(kinds))
print("files in live/ outside data/:", [k.split("live/")[1] for k, _ in listing
                                         if k.startswith(pvc10.LIVE) and not k.startswith(pvc10.LIVE + "data") and not k.endswith("/")])

In [ ]:
P = pvc10.planes()
md5 = pvc10.checksums()
print("planes per mouse:", {m: len(d) for m, d in P.items()}, "total", sum(len(d) for d in P.values()))
print("paper: 6 animals, 6 to 15 focal planes each, 150 to 460 um below the pia")
for m, ds in P.items():
    st = pvc10.stats_table(m)
    same = sorted(st[:, 0].astype(int).tolist()) == ds
    shifts = sorted({(float(a), float(b)) for a, b in st[:, 2:]})
    print(f"mouse{m}: depths {ds[0]} to {ds[-1]} um; stats depths match file names: {same}; "
          f"stimulus duration {sorted(set(st[:, 1]))} s; plane shifts (x, y um) {shifts}")
listed = set(md5)
missing_md5 = [k for k, _ in live_data if k.split("live/")[1] not in listed]
print("live/data files without an MD5 in checksums.md5:", len(missing_md5))

### Stimulus tables

Each stim file has two columns: the frame at which a stimulus starts and its condition (1 blank,
2 to 9 the binocular phases 0 to 315 deg, 10 ipsilateral, 11 contralateral). The paper says 6 to
7 repeats per focal plane, a 1.5 to 2 s stimulus followed by a 3 s blank, and imaging at 30 Hz.

In [ ]:
stims = {}
rows = []
for m, ds in P.items():
    dur = pvc10.stats_table(m)[0, 1]
    for d in ds:
        s = pvc10.stim_table(m, d)
        stims[(m, d)] = s
        counts = np.bincount(s[:, 1], minlength=12)[1:]
        isi = np.diff(s[:, 0])
        rows.append((m, d, len(s), counts.min(), counts.max(), set(s[:, 1]) == set(range(1, 12)),
                     np.median(isi), isi.min(), isi.max(), dur))
print(f"{'mouse':>5s} {'depth':>5s} {'trials':>6s} {'rep min':>7s} {'rep max':>7s} {'11 conds':>8s} "
      f"{'ISI med':>7s} {'min':>4s} {'max':>5s} {'stim s':>6s}")
for r in rows:
    print(f"{r[0]:5d} {r[1]:5d} {r[2]:6d} {r[3]:7d} {r[4]:7d} {str(r[5]):>8s} {r[6]:7.0f} {r[7]:4d} {r[8]:5d} {r[9]:6.1f}")
reps = np.array([[r[3], r[4]] for r in rows])
print("planes with fewer than 6 repeats of some condition:",
      [(r[0], r[1], r[3]) for r in rows if r[3] < 6])
print()
print("Implied frame rate if each cycle is the stimulus plus a 3 s blank (Methods):")
for m in P:
    isi = np.median(np.concatenate([np.diff(stims[(m, d)][:, 0]) for d in P[m]]))
    dur = pvc10.stats_table(m)[0, 1]
    print(f"  mouse{m}: median onset interval {isi:.0f} frames, cycle {dur + 3:.1f} s -> "
          f"{isi / (dur + 3):.1f} frames/s; at 30 Hz the interval is {isi / 30:.2f} s")

### Movies

For every plane, the frame count and frame shape of both channels, the range of the first frame,
and a hash of the first green frame, to look for recordings deposited twice. Only the first frame
of each file is read here.

In [ ]:
t = time.time()
movies = {}
for m, ds in P.items():
    for d in ds:
        for ch in ("green", "red"):
            mv = pvc10.Movie(m, d, ch)
            f0 = mv.frame(0)
            movies[(m, d, ch)] = dict(n=mv.n_frames, shape=f0.shape, dtype=str(mv.frame_dataset(0).dtype),
                                      lo=f0.min(), hi=f0.max(), mean=f0.mean(),
                                      h=hashlib.md5(f0.tobytes()).hexdigest(), img=f0[::4, ::4].copy())
            mv.close()
print(f"read {len(movies)} files in {time.time() - t:.0f} s")
print("frame shapes (rows, columns):", collections.Counter(v["shape"] for v in movies.values()))
print("dtypes:", collections.Counter(v["dtype"] for v in movies.values()))
for ch in ("green", "red"):
    lo = [v["lo"] for k, v in movies.items() if k[2] == ch]
    hi = [v["hi"] for k, v in movies.items() if k[2] == ch]
    mn = [v["mean"] for k, v in movies.items() if k[2] == ch]
    print(f"{ch}: first-frame minimum {min(lo):.0f} to {max(lo):.0f}, maximum {min(hi):.0f} to {max(hi):.0f}, "
          f"mean {min(mn):.1f} to {max(mn):.1f}")
mismatch = [(m, d) for m, ds in P.items() for d in ds if movies[(m, d, 'green')]['n'] != movies[(m, d, 'red')]['n']]
print("planes whose green and red frame counts differ:", mismatch)
short = [(m, d) for m, ds in P.items() for d in ds
         if stims[(m, d)][-1, 0] + np.median(np.diff(stims[(m, d)][:, 0])) > movies[(m, d, 'green')]['n']]
print("planes whose last stimulus cycle runs past the last frame:", short)
nf = [movies[(m, d, 'green')]['n'] for m, ds in P.items() for d in ds]
print("frames per plane:", min(nf), "to", max(nf))
hashes = collections.Counter(v["h"] for k, v in movies.items())
print("first frames identical between files:", [h for h, c in hashes.items() if c > 1])
# similarity of first green frames between planes of the same mouse
for m, ds in P.items():
    imgs = np.array([movies[(m, d, 'green')]['img'].ravel() for d in ds])
    c = np.corrcoef(imgs)
    np.fill_diagonal(c, np.nan)
    print(f"mouse{m}: highest correlation of first green frames between two planes {np.nanmax(c):.3f}")
print("descriptor: frames of 255 by 456 pixels; paper: 256 by 455 pixels; 300 um field")

### What the published files hold per cell

The paper's cells "were identified by hand from structure images based on size, shape, and
brightness" and PV+ cells "by selecting OGB-1-filled neurons with co-localized tdTomato
fluorescence" (Experimental Procedures, Data Imaging Analysis). Every claim needs those cells,
their masks and their PV+/PV- labels. The only masks in the published folder are the example ones
in `live/script/cellmasks.mat`.

In [ ]:
cm = pvc10.load_small_mat(pvc10.LIVE + "script/cellmasks.mat")
for k, v in cm.items():
    print(k, v.shape, v.dtype, "values", np.unique(v).tolist(), "pixels per mask",
          int(v.sum(axis=(0, 1)).min()), "to", int(v.sum(axis=(0, 1)).max()))
print("files in live/ whose names suggest per-cell data (mask, cell, roi, binoc, peak, dsi):",
      [k.split("live/")[1] for k, _ in listing if k.startswith(pvc10.LIVE) and not k.endswith("/")
       and any(w in k.rsplit("/", 1)[1].lower() for w in ("mask", "cell", "roi", "binoc", "peak", "dsi"))])

The folder `original/to_remove/`, outside the published folder, holds MATLAB structs from
November 2014. They are listed here, to describe the finding; no claim uses them.

In [ ]:
t = time.time()
import fsspec, h5py
for k, s in pvc10.list_bucket(pvc10.DATASET + "/original/"):
    if not k.endswith(".mat"):
        continue
    name = k.rsplit("/", 1)[1]
    if name.startswith("binocmonocinds"):
        d = pvc10.load_small_mat(k)
        print(f"{name:28s} {s:>11d} B  indices: {d['inds'].size}")
        continue
    try:
        fh = fsspec.open(pvc10.BUCKET + k, "rb", block_size=2**20).open()
        h = h5py.File(fh, "r")
    except OSError as e:
        print(f"{name:28s} {s:>11d} B  cannot be opened: {e}")
        continue
    for top in [x for x in h.keys() if not x.startswith("#")]:
        g = h[top]
        fields = list(g.keys())
        deps = collections.Counter(float(h[r][()].ravel()[0]) for r in g["depth"][:, 0])
        print(f"{name:28s} {s:>11d} B  struct {top}: {g[fields[0]].shape[0]} entries, depths {sorted(deps)}")
    h.close()
print("fields:", fields)
print(f"({time.time() - t:.0f} s)")

### The clock of the stimulus table, on the example plane

The descriptor's example plane (mouse5, depth 260 um) is the only one with masks. Its green movie
is streamed whole, the mean intensity in each of the 50 example masks is taken per frame (minus the
offset of -160 that the data set's `examplecode.m` subtracts), and dF/F per trial is computed with
Fo the 10 frames before onset (a standard choice; see the helper). If the stimulus table shares the
movie's clock, stimulus trials should differ from blank trials in a window after onset, and the
difference should fall when the onsets are moved or the condition labels shuffled. This checks one
plane of 58; the others are named as not checked.

In [ ]:
M, D = 5, 260
masks = cm["cellmasks"]
mv = pvc10.Movie(M, D, "green")
traces, secs = pvc10.mask_traces(mv, masks)
mv.close()
stim = stims[(M, D)]
size_gb = dict(live_data)[pvc10.plane_path(M, D, "green")] / 1e9
print(f"streamed {traces.shape[1]} frames ({size_gb:.2f} GB) in {secs:.0f} s, "
      f"{size_gb * 1e3 / secs:.1f} MB/s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
print("raw trace range", round(traces.min(), 1), "to", round(traces.max(), 1))

N_POST = 118  # one onset interval of this plane (median 118 frames)
x = pvc10.trial_dff(traces, stim, N_POST)
lab = stim[:, 1]
avg_stim = np.nanmean(x[:, lab != 1], axis=(0, 1)) * 100
avg_blank = np.nanmean(x[:, lab == 1], axis=(0, 1)) * 100
peak = int(np.argmax(avg_stim))
print(f"mean dF/F over cells and stimulus trials peaks at frame {peak} after onset, {avg_stim[peak]:.2f}%; "
      f"blank trials at that frame {avg_blank[peak]:.2f}%")
print("mean dF/F (%) every 6 frames, stimulus trials:", np.round(avg_stim[::6], 2).tolist())
print("mean dF/F (%) every 6 frames, blank trials:   ", np.round(avg_blank[::6], 2).tolist())

# the paper's 0.5 s window, centred on the average peak (Methods); 0.5 s is 15 frames at 30 Hz
half = 7
win = slice(max(peak - half, 0), peak + half + 1)


def contrast(shift=0, labels=None):
    xx = pvc10.trial_dff(traces, stim, N_POST, shift)
    r = np.nanmean(xx[:, :, win], axis=2)
    ll = lab if labels is None else labels
    return (np.nanmean(r[:, ll != 1]) - np.nanmean(r[:, ll == 1])) * 100, r


c0, R = contrast()
rng = np.random.default_rng(20150630)
null = np.array([contrast(0, rng.permutation(lab))[0] for _ in range(200)])
print(f"stimulus minus blank in the window (frames {win.start} to {win.stop - 1}): {c0:.2f}% dF/F; "
      f"with shuffled labels 95% range {np.percentile(null, 2.5):.2f} to {np.percentile(null, 97.5):.2f}, "
      f"p = {(np.sum(np.abs(null) >= abs(c0)) + 1) / (len(null) + 1):.3f}")
shifts = [-118, -90, -60, -30, 0, 30, 60, 90]
cs = [contrast(s)[0] for s in shifts]
for s, c in zip(shifts, cs):
    print(f"  onsets moved by {s:+4d} frames: stimulus minus blank {c:6.2f}%")

# responsiveness index of the Methods per mask, without the neuropil requirement
n = np.array([np.sum(lab == k) for k in range(1, 12)])
mb, sb = R[:, lab == 1].mean(1), R[:, lab == 1].std(1, ddof=1) / np.sqrt(n[0])
idx = np.stack([(R[:, lab == k].mean(1) - mb) / (R[:, lab == k].std(1, ddof=1) / np.sqrt(n[k - 1]) + sb)
                for k in range(2, 12)], 1)
print("example masks with index >= 1 for some binocular condition:", int((idx[:, :8] >= 1).any(1).sum()),
      "of 50; for some monocular condition:", int((idx[:, 8:] >= 1).any(1).sum()),
      "(the descriptor calls them a mixture of responsive and non-responsive cells)")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(avg_stim, label="stimulus trials")
ax[0].plot(avg_blank, label="blank trials")
ax[0].axvspan(win.start, win.stop - 1, color="0.9")
ax[0].set_xlabel("frames after onset"); ax[0].set_ylabel("mean dF/F (%)"); ax[0].legend()
ax[1].plot(shifts, cs, "o-")
ax[1].axhspan(np.percentile(null, 2.5), np.percentile(null, 97.5), color="0.9")
ax[1].set_xlabel("onset shift (frames)"); ax[1].set_ylabel("stimulus minus blank (%)")
plt.tight_layout(); plt.show()
print(f"plotted ranges: left {min(avg_stim.min(), avg_blank.min()):.2f} to {max(avg_stim.max(), avg_blank.max()):.2f}%; "
      f"right {min(cs):.2f} to {max(cs):.2f}%; nothing outside the axes")

### Red channel of the example masks

The descriptor says the red channel "can be used to identify if the cell is parvalbumin positive or
not", but gives no labels. The mean red intensity in each example mask over the first 160 frames
(the number `examplecode.m` averages) shows what a reader would have to threshold, with no rule from
the paper or the data set to do it.

In [ ]:
mv = pvc10.Movie(M, D, "red")
red, rsecs = pvc10.mask_traces(mv, masks, range(160))
red_img = np.mean([mv.frame(i) for i in range(0, 160, 8)], axis=0)
mv.close()
r = red.mean(1)
print(f"red mean in the 50 masks: {r.min():.1f} to {r.max():.1f} (median {np.median(r):.1f}); "
      f"whole-frame median {np.median(red_img) - pvc10.DARKNOISE:.1f}")
print("masks sorted by red intensity:", np.round(np.sort(r), 0).astype(int).tolist())

### Size of a full re-analysis

Re-running the paper's analysis would need the cells identified anew in all 58 planes (by hand, in
the paper) and both channels streamed. The rate measured above gives the time to stream the movies
alone.

In [ ]:
movie_bytes = sum(s for k, s in live_data if k.endswith("green.mat") or k.endswith("red.mat"))
rate = size_gb * 1e9 / secs
print(f"{sum(1 for k, _ in live_data if k.endswith(('green.mat', 'red.mat')))} movies, {movie_bytes / 1e9:.0f} GB; "
      f"at {rate / 1e6:.1f} MB/s, {movie_bytes / rate / 3600:.1f} h to stream once, against a 1 h notebook limit")
print(f"notebook time so far {(time.time() - T0) / 60:.1f} min")

## Claims

No claim is tested. Every finding of the paper rests on its hand-identified cells (4,320 in the
six PV-Cre mice), their PV+/PV- labels, the CaMKIIa mice, the neuropil patches or the histology,
and none of these is in the published data set. The reasons are in the table in the header and in
the attempt record.


## Summary

| Claim | Outcome |
|---|---|
| fig1e-pv-tdtomato-colocalization | blocked, data_missing (histology) |
| cell-counts-responsive, fig3abc-pv-more-binocular-fraction, fig3c-contralateral-bias, fig3-pv-lower-monocularity, mi-per-animal, fig3def-pv-weaker-disparity-selectivity, mi-dsi-relation, pv-larger-dff-low-dsi | blocked, data_missing (cell masks and PV labels) |
| fig4ag-salt-and-pepper, fig4d-pv-od-matches-local-50um, fig4f-pv-od-spatial-decay, fig4f-od-length-constant, fig4ef-pvminus-od-unrelated, figs6ab-pvminus-pv-like-od, figs4-same-plane-control, fig4i-pv-disparity-matches-local, fig4k-pv-disparity-length-constant, fig4jk-pvminus-disparity-weak, figs6cd-pvminus-pv-like-disparity | blocked, data_missing (cell masks and PV labels) |
| figs3-pv-distinct-from-neuropil, figs5-neuropil-no-relation | blocked, data_missing (cells and neuropil patches) |
| fig5-camkii-responsive, fig5e-camkii-od-unrelated, fig5f-camkii-disparity-unrelated | blocked, data_missing (CaMKIIa animals) |

How the analysis differs from the paper's: no analysis of the paper was run. The data check reads
every stimulus and stats file and the first frame of every movie, and streams one plane with the
data set's 50 example masks.

What was found about the data set, from the outputs above:

- The published folder holds raw movies (green and red), stimulus tables and plane shifts for 58
  planes of 6 mice, and 50 example masks for one plane. It holds none of the per-cell data that its
  descriptor says it contains (fluorescence traces, peak responses per condition, DSI, preferred
  disparity, cell positions), and no masks or PV+/PV- labels for the paper's cells.
- The CaMKIIa-nls-tdTomato mice, the neuropil patches and the anti-PV histology are not in it.
- A folder outside the published one, `original/to_remove/`, holds processed per-cell structs
  (masks, peaks, DSI, disparity preference, positions, neuropil responses) for mice 7 to 10 only;
  three of the mouse 7 files are truncated and cannot be opened.
- The descriptor gives frames of 255 by 456 pixels; the files hold 256 by 455, as the paper says.
- No frame rate is given in the data set. At the paper's 30 Hz, the median onset intervals give
  cycles of 3.93 s in mouse 5 and 4.37 to 4.43 s in mice 7 to 10, against the 4.5 s and 5 s of the
  stimulus plus a 3 s blank in the Methods; mouse 6 (4.43 s against 4.5 s) fits. Either the frame
  rate or the blank differs from the Methods in five of six mice.
- In mouse 5 four planes are shifted by 300 um in x and -300 um in y, and in mouse 7 the planes
  from 240 um down by 157 and -70 um: a second field of view, which the files name only by these
  shifts.
- In four planes the last stimulus cycle runs past the last frame.
- Eight planes have fewer than the 6 to 7 repeats the paper states for some condition (all six
  planes of mouse 10 have 4 or 5).
- No two movies share a first frame, and the first green frames of two planes of one mouse
  correlate at most 0.52 to 0.76, so no recording was found deposited twice.
- On the example plane, stimulus trials differ from blank trials after onset, and the difference
  falls when onsets are moved, so the stimulus table is consistent with the movie's clock there.
